# Car Sales – Full EDA + Regression Chart Pack
Dataset: `car_sales_regression_500_rows.xlsx` (500 rows × 14 columns)  
Target for regression: **Sales_Price_Lakh**

Every chart is displayed in the notebook **and** saved as a page in `car_sales_eda_regression_charts.pdf`.

**Requirements:** `pip install pandas numpy matplotlib seaborn scipy scikit-learn openpyxl`  
Keep the Excel file in the same folder as this notebook, then run *Kernel → Restart & Run All*.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.backends.backend_pdf import PdfPages

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold", "axes.titlesize": 11})

DATA_PATH = "car_sales_regression_500_rows.xlsx"
PDF_PATH  = "car_sales_eda_regression_charts.pdf"
pdf = PdfPages(PDF_PATH)

def save(fig, title=None):
    """Add a page to the PDF and show the figure in the notebook."""
    if title:
        fig.suptitle(title, fontsize=15, fontweight="bold", y=0.995)
        fig.tight_layout(rect=[0, 0, 1, 0.96])
    else:
        fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 1. Load data & basic checks

In [ ]:
df = pd.read_excel(DATA_PATH)
TARGET   = "Sales_Price_Lakh"
num_cols = ["Age_Years", "KM_Driven", "Engine_CC", "Owner_Count", "Mileage_KMPL",
            "Original_Price_Lakh", "Discount_Percent", "Units_Sold"]
cat_cols = ["Brand", "Fuel_Type", "Transmission", "City"]

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()), "| Duplicate rows:", int(df.duplicated().sum()),
      "| Duplicate Car_IDs:", int(df["Car_ID"].duplicated().sum()))
df.head()

In [ ]:
# Cover page + summary-statistics table
fig, ax = plt.subplots(figsize=(11, 8)); ax.axis("off")
ax.text(0.5, 0.90, "Car Sales – EDA & Regression Charts", ha="center", fontsize=26, fontweight="bold")
ax.text(0.5, 0.84, f"{df.shape[0]} rows × {df.shape[1]} columns   |   Target: {TARGET}",
        ha="center", fontsize=13, color="dimgray")
contents = [
    "1. Summary statistics", "2. Target distribution (hist, box, Q-Q)", "3. Histograms of all numeric features",
    "4. Box plots – outlier check", "5. Categorical counts", "6. Correlation heatmaps (Pearson & Spearman)",
    "7. Correlation with target", "8. Feature vs target scatter + regression line",
    "9. Target by category (box plots)", "10. Mean price heatmaps", "11. Multi-variable scatter plots",
    "12. Pair plot", "13. Price retention / depreciation", "14. Units sold analysis",
    "15. Multicollinearity (VIF)", "16. Model comparison (R², RMSE, MAE)", "17. Actual vs predicted",
    "18. Residual diagnostics", "19. Feature importance / coefficients", "20. Cross-validation",
    "21. Leakage check (with vs without price/discount)"]
for i, line in enumerate(contents):
    ax.text(0.12 if i < 11 else 0.55, 0.74 - (i % 11) * 0.055, line, fontsize=11)
save(fig)

desc = df[num_cols + [TARGET]].describe().T.round(2)
fig, ax = plt.subplots(figsize=(12, 4.5)); ax.axis("off")
tbl = ax.table(cellText=desc.values, colLabels=desc.columns, rowLabels=desc.index, loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
save(fig, "1. Summary statistics (numeric columns)")

## 2. Target distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sns.histplot(df[TARGET], kde=True, bins=30, ax=axes[0], color="steelblue")
axes[0].axvline(df[TARGET].mean(), color="crimson", ls="--", label=f"mean = {df[TARGET].mean():.2f}")
axes[0].axvline(df[TARGET].median(), color="green", ls="--", label=f"median = {df[TARGET].median():.2f}")
axes[0].legend(); axes[0].set_title(f"Histogram + KDE (skew = {df[TARGET].skew():.2f})")
sns.boxplot(y=df[TARGET], ax=axes[1], color="lightsteelblue"); axes[1].set_title("Box plot")
stats.probplot(df[TARGET], dist="norm", plot=axes[2]); axes[2].set_title("Normal Q-Q plot")
save(fig, "2. Target distribution – Sales_Price_Lakh")

## 3. Distributions of numeric features

In [ ]:
cols = num_cols + [TARGET]
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, c in zip(axes.ravel(), cols):
    sns.histplot(df[c], kde=df[c].nunique() > 10, bins=min(30, df[c].nunique()), ax=ax, color="steelblue")
    ax.set_title(f"{c} (skew {df[c].skew():.2f})")
save(fig, "3. Histograms of numeric features")

## 4. Outlier check (box plots)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, c in zip(axes.ravel(), cols):
    sns.boxplot(x=df[c], ax=ax, color="lightsteelblue", fliersize=3)
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    n_out = int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum())
    ax.set_title(f"{c}  ({n_out} outliers)")
save(fig, "4. Box plots – outlier check")

## 5. Categorical variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, c in zip(axes.ravel()[:2], ["Brand", "City"]):
    order = df[c].value_counts().index
    sns.countplot(data=df, y=c, order=order, ax=ax, color="steelblue"); ax.set_title(f"{c} counts")
sns.countplot(data=df, x="Fuel_Type", order=df["Fuel_Type"].value_counts().index, ax=axes[0, 2], color="steelblue")
axes[0, 2].set_title("Fuel_Type counts")
df["Transmission"].value_counts().plot.pie(ax=axes[1, 0], autopct="%1.1f%%", startangle=90,
                                            colors=sns.color_palette("pastel")); axes[1, 0].set_ylabel("")
axes[1, 0].set_title("Transmission share")
sns.countplot(data=df, x="Owner_Count", ax=axes[1, 1], color="steelblue"); axes[1, 1].set_title("Owner_Count counts")
sns.countplot(data=df, x="Age_Years", ax=axes[1, 2], color="steelblue"); axes[1, 2].set_title("Age_Years counts")
save(fig, "5. Categorical variables")

## 6. Correlations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
for ax, method in zip(axes, ["pearson", "spearman"]):
    corr = df[cols].corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                square=True, linewidths=.5, cbar_kws={"shrink": .75}, ax=ax)
    ax.set_title(f"{method.title()} correlation")
save(fig, "6. Correlation heatmaps")

corr_t = df[num_cols].corrwith(df[TARGET]).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["crimson" if v < 0 else "seagreen" for v in corr_t]
ax.barh(corr_t.index, corr_t.values, color=colors)
for i, v in enumerate(corr_t.values):
    ax.text(v + (0.01 if v >= 0 else -0.01), i, f"{v:.2f}", va="center", ha="left" if v >= 0 else "right")
ax.axvline(0, color="black", lw=.8); ax.set_xlabel("Pearson r with Sales_Price_Lakh"); ax.set_xlim(-1.1, 1.1)
save(fig, "7. Correlation of each feature with the target")

## 7. Feature vs target (scatter + regression line)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, c in zip(axes.ravel(), num_cols):
    jitter = 0.15 if df[c].nunique() <= 10 else None
    sns.regplot(data=df, x=c, y=TARGET, ax=ax, x_jitter=jitter,
                scatter_kws=dict(alpha=.35, s=14), line_kws=dict(color="crimson"))
    ax.set_title(f"{c}  (r = {df[c].corr(df[TARGET]):.2f})")
save(fig, "8. Numeric features vs Sales_Price_Lakh")

## 8. Target by category

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, c in zip(axes.ravel(), ["Brand", "City", "Fuel_Type", "Transmission", "Owner_Count", "Age_Years"]):
    order = df.groupby(c)[TARGET].median().sort_values().index if c in cat_cols else sorted(df[c].unique())
    sns.boxplot(data=df, x=c, y=TARGET, order=order, ax=ax, color=sns.color_palette()[0], fliersize=3)
    if c in ("Brand", "City"): ax.tick_params(axis="x", rotation=45)
    ax.set_title(f"Sales price by {c}")
save(fig, "9. Sales_Price_Lakh by category")

## 9. Mean price heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.heatmap(df.pivot_table(index="Brand", columns="Fuel_Type", values=TARGET, aggfunc="mean"),
            annot=True, fmt=".1f", cmap="YlGnBu", ax=axes[0]); axes[0].set_title("Mean price: Brand × Fuel_Type")
sns.heatmap(df.pivot_table(index="City", columns="Transmission", values=TARGET, aggfunc="mean"),
            annot=True, fmt=".1f", cmap="YlGnBu", ax=axes[1]); axes[1].set_title("Mean price: City × Transmission")
sns.heatmap(df.pivot_table(index="Age_Years", columns="Owner_Count", values=TARGET, aggfunc="mean"),
            annot=True, fmt=".1f", cmap="YlGnBu", ax=axes[2]); axes[2].set_title("Mean price: Age × Owner_Count")
save(fig, "10. Mean Sales_Price_Lakh heatmaps")

## 10. Multi-variable relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
sns.scatterplot(data=df, x="Age_Years", y=TARGET, hue="Fuel_Type", alpha=.6, ax=axes[0, 0]); axes[0, 0].set_title("Age vs price by Fuel_Type")
sns.scatterplot(data=df, x="KM_Driven", y=TARGET, hue="Transmission", alpha=.6, ax=axes[0, 1]); axes[0, 1].set_title("KM driven vs price by Transmission")
sc = axes[1, 0].scatter(df["Original_Price_Lakh"], df[TARGET], c=df["Age_Years"], cmap="viridis", alpha=.7, s=20)
lim = [df[["Original_Price_Lakh", TARGET]].min().min(), df["Original_Price_Lakh"].max()]
axes[1, 0].plot(lim, lim, "r--", label="no depreciation (y = x)"); axes[1, 0].legend()
plt.colorbar(sc, ax=axes[1, 0], label="Age_Years")
axes[1, 0].set_xlabel("Original_Price_Lakh"); axes[1, 0].set_ylabel(TARGET); axes[1, 0].set_title("Original vs sales price (colour = age)")
sns.scatterplot(data=df, x="Mileage_KMPL", y=TARGET, hue="Fuel_Type", alpha=.6, ax=axes[1, 1]); axes[1, 1].set_title("Mileage vs price by Fuel_Type")
save(fig, "11. Multi-variable scatter plots")

## 11. Pair plot

In [ ]:
pair_cols = ["Age_Years", "KM_Driven", "Original_Price_Lakh", "Discount_Percent", "Mileage_KMPL", TARGET]
g = sns.pairplot(df[pair_cols], corner=True, diag_kind="kde", plot_kws=dict(alpha=.35, s=12))
g.figure.suptitle("12. Pair plot of key variables", fontsize=15, fontweight="bold", y=1.02)
pdf.savefig(g.figure, bbox_inches="tight"); plt.show(); plt.close(g.figure)

## 12. Price retention (sales price ÷ original price)

In [ ]:
d2 = df.assign(Price_Retention=df[TARGET] / df["Original_Price_Lakh"])
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.lineplot(data=d2, x="Age_Years", y="Price_Retention", hue="Fuel_Type", marker="o", ax=axes[0, 0])
axes[0, 0].set_title("Retention vs Age by Fuel_Type (mean ± 95% CI)")
sns.regplot(data=d2, x="Discount_Percent", y="Price_Retention", ax=axes[0, 1],
            scatter_kws=dict(alpha=.35, s=14), line_kws=dict(color="crimson")); axes[0, 1].set_title("Retention vs Discount_Percent")
order = d2.groupby("Brand")["Price_Retention"].median().sort_values().index
sns.boxplot(data=d2, x="Brand", y="Price_Retention", order=order, ax=axes[1, 0], color=sns.color_palette()[0])
axes[1, 0].tick_params(axis="x", rotation=45); axes[1, 0].set_title("Retention by Brand")
sns.histplot(d2["Price_Retention"], kde=True, ax=axes[1, 1], color="steelblue"); axes[1, 1].set_title("Retention distribution")
save(fig, "13. Price retention / depreciation")

## 13. Units sold

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.boxplot(data=df, x="Brand", y="Units_Sold", ax=axes[0, 0], color=sns.color_palette()[0]); axes[0, 0].tick_params(axis="x", rotation=45)
axes[0, 0].set_title("Units sold by Brand")
sns.boxplot(data=df, x="City", y="Units_Sold", ax=axes[0, 1], color=sns.color_palette()[1]); axes[0, 1].tick_params(axis="x", rotation=45)
axes[0, 1].set_title("Units sold by City")
sns.boxplot(data=df, x="Fuel_Type", y="Units_Sold", ax=axes[1, 0], color=sns.color_palette()[2]); axes[1, 0].set_title("Units sold by Fuel_Type")
sns.regplot(data=df, x="Discount_Percent", y="Units_Sold", ax=axes[1, 1], scatter_kws=dict(alpha=.35, s=14), line_kws=dict(color="crimson"))
axes[1, 1].set_title(f"Units sold vs Discount (r = {df['Discount_Percent'].corr(df['Units_Sold']):.2f})")
save(fig, "14. Units sold analysis")

## 14. Regression: multicollinearity check (VIF)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

def vif(X):
    out = {}
    for c in X.columns:
        others = X.drop(columns=c)
        r2 = LinearRegression().fit(others, X[c]).score(others, X[c])
        out[c] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(out).sort_values()

v = vif(df[num_cols])
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(v.index, v.values, color="steelblue")
ax.axvline(5, color="orange", ls="--", label="VIF = 5"); ax.axvline(10, color="crimson", ls="--", label="VIF = 10")
for i, val in enumerate(v.values): ax.text(val + .02, i, f"{val:.2f}", va="center")
ax.legend(); ax.set_xlabel("Variance Inflation Factor")
save(fig, "15. Multicollinearity (VIF) – numeric predictors")

## 15. Fit models

In [ ]:
y = df[TARGET]
X_all = pd.get_dummies(df.drop(columns=["Car_ID", TARGET]), columns=cat_cols, drop_first=True).astype(float)
X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge":             make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Random Forest":     RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}
preds, rows = {}, []
for name, m in models.items():
    m.fit(X_train, y_train); p = m.predict(X_test); preds[name] = p
    rows.append({"Model": name, "R2": r2_score(y_test, p),
                 "RMSE": np.sqrt(mean_squared_error(y_test, p)), "MAE": mean_absolute_error(y_test, p)})
res = pd.DataFrame(rows).set_index("Model")
print(res.round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, metric in zip(axes, ["R2", "RMSE", "MAE"]):
    sns.barplot(x=res.index, y=res[metric], ax=ax, hue=res.index, legend=False, palette="deep")
    for i, val in enumerate(res[metric]): ax.text(i, val, f"{val:.3f}", ha="center", va="bottom")
    ax.tick_params(axis="x", rotation=20); ax.set_xlabel(""); ax.set_title(metric)
save(fig, "16. Model comparison on the 20% hold-out set")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, (name, p) in zip(axes.ravel(), preds.items()):
    ax.scatter(y_test, p, alpha=.6, s=25)
    lim = [min(y_test.min(), p.min()), max(y_test.max(), p.max())]
    ax.plot(lim, lim, "r--"); ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
    ax.set_title(f"{name}  (R² = {res.loc[name, 'R2']:.3f})")
save(fig, "17. Actual vs predicted")

best = res["R2"].idxmax()
resid = y_test.values - preds[best]
std_resid = (resid - resid.mean()) / resid.std()
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes[0, 0].scatter(preds[best], resid, alpha=.6, s=25); axes[0, 0].axhline(0, color="red", ls="--")
axes[0, 0].set_xlabel("Predicted"); axes[0, 0].set_ylabel("Residual"); axes[0, 0].set_title("Residuals vs predicted")
sns.histplot(resid, kde=True, ax=axes[0, 1], color="steelblue"); axes[0, 1].set_title("Residual distribution")
stats.probplot(resid, dist="norm", plot=axes[1, 0]); axes[1, 0].set_title("Residual Q-Q plot")
axes[1, 1].scatter(preds[best], np.sqrt(np.abs(std_resid)), alpha=.6, s=25)
axes[1, 1].set_xlabel("Predicted"); axes[1, 1].set_ylabel("√|standardised residual|"); axes[1, 1].set_title("Scale-location (constant variance check)")
save(fig, f"18. Residual diagnostics – best model: {best}")

In [ ]:
lin = models["Linear Regression"]
coef = pd.Series(lin[-1].coef_, index=X_all.columns).sort_values()
top = pd.concat([coef.head(8), coef.tail(8)]).drop_duplicates()

rf = models["Random Forest"]
perm = permutation_importance(rf, X_test, y_test, n_repeats=15, random_state=42)
perm_s = pd.Series(perm.importances_mean, index=X_all.columns).sort_values().tail(12)
rf_imp = pd.Series(rf.feature_importances_, index=X_all.columns).sort_values().tail(12)

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].barh(top.index, top.values, color=["crimson" if v < 0 else "seagreen" for v in top.values])
axes[0].axvline(0, color="black", lw=.8); axes[0].set_title("Linear regression – standardised coefficients (top 16)")
axes[1].barh(rf_imp.index, rf_imp.values, color="steelblue"); axes[1].set_title("Random Forest – impurity importance (top 12)")
axes[2].barh(perm_s.index, perm_s.values, color="darkorange"); axes[2].set_title("Random Forest – permutation importance (top 12)")
save(fig, "19. Feature importance / coefficients")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv = {n: cross_val_score(m, X_all, y, cv=kf, scoring="r2") for n, m in models.items()}
cv_df = pd.DataFrame(cv)
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.boxplot(data=cv_df, ax=ax, palette="deep"); sns.stripplot(data=cv_df, ax=ax, color="black", size=5)
ax.set_ylabel("R² (5-fold CV)"); ax.tick_params(axis="x", rotation=15)
save(fig, "20. Cross-validated R² by model")

## 16. Leakage check – do Original_Price_Lakh and Discount_Percent dominate?

In [ ]:
def eval_features(feat_df):
    Xtr, Xte, ytr, yte = train_test_split(feat_df, y, test_size=0.2, random_state=42)
    out = {}
    for n, m in models.items():
        from sklearn.base import clone
        mm = clone(m).fit(Xtr, ytr); out[n] = r2_score(yte, mm.predict(Xte))
    return out

drop_cols = [c for c in X_all.columns if c in ("Original_Price_Lakh", "Discount_Percent")]
comp = pd.DataFrame({"All features": eval_features(X_all),
                     "Without Original_Price & Discount": eval_features(X_all.drop(columns=drop_cols))})
fig, ax = plt.subplots(figsize=(10, 5.5))
comp.plot.bar(ax=ax, color=["steelblue", "darkorange"]); ax.set_ylabel("Test R²"); ax.tick_params(axis="x", rotation=15)
ax.set_ylim(0, 1.15); ax.legend(loc="upper center", ncol=2)
for cont in ax.containers: ax.bar_label(cont, fmt="%.2f", padding=2)
save(fig, "21. R² with vs without Original_Price & Discount")

pdf.close()
print("Saved:", PDF_PATH)